In [1]:
import os
import scipy.io
import json
import re
import numpy as np
from pyemittance.emittance_calc import EmitCalc

In [2]:
scans_dir = f"matlab_scans/"
regex = r"^Emittance-scan-OTRS_IN20_571-202[0-9]-[0-9]{2}-[0-9]{2}-[0-9]{6}\.mat$"


filenames=[]
for file in os.scandir(scans_dir):
    if re.match(regex, file.name):
        filenames.append(file.name)
print('Total ',len(filenames),'files')

Total  51 files


In [3]:
fname = os.path.join("LCLS_OTR2", "beamline_info" + ".json")
config = {"beamline_info": json.load(open(fname))}

In [6]:
for fname in filenames:
    print(fname)
    mat = scipy.io.loadmat(scans_dir + fname)
    mat_data = mat['data'][0]

    num_steps = mat_data['beam'][0].shape[0]
    quadvals = mat_data['quadVal'][0].flatten()

    xrms, yrms, xrms_err, yrms_err = [], [], [], []
    for j in range(num_steps):
        beam_j = mat_data['beam'][0][j, 0]   # method 0
        xrms.append(beam_j['stats'][0, 2])
        yrms.append(beam_j['stats'][0, 3])
        xrms_err.append(beam_j['statsStd'][0, 2])
        yrms_err.append(beam_j['statsStd'][0, 3])

    xrms, yrms = np.array(xrms)*1e-6, np.array(yrms)*1e-6
    xrms_err, yrms_err = np.array(xrms_err)*1e-6, np.array(yrms_err)*1e-6

    try:
        twiss = mat_data['twiss'][0].flatten()
    except (KeyError, IndexError, ValueError):
        print(f"Error in file {fname}.\nSkipping.\n")
        continue
    twissstd = mat_data['twissstd'][0].flatten()
    emitx = twiss[8]
    emitx_err = twissstd[8]
    bmagx = twiss[11]
    bmagx_err = twissstd[11]
    emity = twiss[12]
    emity_err = twissstd[12]
    bmagy = twiss[15]
    bmagy_err = twissstd[15]

    twiss0 = mat_data['twiss0'][0].flatten()

    # pass this info to the Emittance Calc class to start a measurement
    ef = EmitCalc({'x': quadvals, 'y': quadvals},
                  {'x': xrms, 'y': yrms},
                  {'x': xrms_err, 'y': yrms_err},
                  config_dict = config
                  )

    ef.plot = False
    ef.calc_bmag = True

    # get normalized transverse emittance, the out_dict is the results returned in a dictionary
    out_dict = ef.get_emit()

    if out_dict["error_x"] or out_dict["error_y"]:
        print(f"Error in file {fname}.\nSkipping.\n")
        continue

    print("\033[1;3mpyemittance\033[0m")
    print(f"norm_emitx: {out_dict['norm_emit_x']/1e-6:.2f} +/- {out_dict['norm_emit_x_err']/1e-6:.2f} um")
    print(f"norm_emity: {out_dict['norm_emit_y']/1e-6:.2f} +/- {out_dict['norm_emit_y_err']/1e-6:.2f} um")
    print(f"bmagx: {out_dict['screen_bmagx']:.2f} +/- {out_dict['screen_bmagx_err']:.2f}")
    print(f"bmagy: {out_dict['screen_bmagy']:.2f} +/- {out_dict['screen_bmagy_err']:.2f}")

    print("\033[1;3mmatlab\033[0m")
    print(f"norm_emitx: {emitx/1e-6:.2f} +/- {emitx_err/1e-6:.2f} um")
    print(f"norm_emity: {emity/1e-6:.2f} +/- {emity_err/1e-6:.2f} um")
    print(f"bmagx: {bmagx:.2f} +/- {bmagx_err:.2f}")
    print(f"bmagy: {bmagy:.2f} +/- {bmagy_err:.2f}")

    print("\033[1;3mconfigs\033[0m")
    print("PyEmit Twiss0: ", ["%E" % e for e in ef.config_dict['beamline_info']['Twiss0']])
    print("Matlab Twiss0: ", ["%E" % e for e in twiss0])

    print("PyEmit r-matx: ",ef.config_dict['beamline_info']['rMatx'])
    print("PyEmit r-maty: ",ef.config_dict['beamline_info']['rMaty'])
    print("\n")

    #break


Emittance can't be computed. Returning error


Emittance-scan-OTRS_IN20_571-2021-02-18-125147.mat
pyemittance
norm_emitx: 0.61 +/- 0.01 um
norm_emity: 0.33 +/- 0.01 um
bmagx: 1.03 +/- 0.01
bmagy: 1.17 +/- 0.03
matlab
norm_emitx: 0.35 +/- 0.04 um
norm_emity: 0.40 +/- 0.02 um
bmagx: 0.00 +/- 0.00
bmagy: 1.08 +/- 0.03
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', '1.000000E-06', '1.110117E+00', '1.110117E+00', '-6.877059E-02', '-6.877052E-02']
PyEmit r-matx:  [1, 2.26, 0, 1]
PyEmit r-maty:  [1, 2.26, 0, 1]


Emittance-scan-OTRS_IN20_571-2021-05-01-114946.mat
pyemittance
norm_emitx: 0.70 +/- 0.01 um
norm_emity: 0.57 +/- 0.01 um
bmagx: 1.22 +/- 0.02
bmagy: 1.38 +/- 0.02
matlab
norm_emitx: 0.64 +/- 0.01 um
norm_emity: 0.65 +/- 0.03 um
bmagx: 0.00 +/- 0.00
bmagy: 0.72 +/- 0.09
configs
PyEmit Twiss0:  ['1.000000E-06', '1.000000E-06', '1.113081E+00', '1.113022E+00', '-6.894036E-02', '-7.029490E-02']
Matlab Twiss0:  ['1.000000E-06', 